In [2]:
import pandas as pd
import numpy as np
import os

# ============================================================
# LOAD SAS DATA FILE FROM YOUR LOCAL PATH
# ============================================================

sas_path = '/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/data/raw/HC2024_SAS.sas7bdat'

print(f"Loading: {sas_path}")
print(f"File exists: {os.path.exists(sas_path)}")

# Read SAS file
df = pd.read_sas(sas_path, encoding='latin1')
print(f"✓ Successfully loaded SAS file")

# ============================================================
# INITIAL DATA INSPECTION
# ============================================================

print("\n" + "="*80)
print("NAMCS HC 2024 DATA - INITIAL INSPECTION")
print("="*80)

print(f"\nDataset Shape: {df.shape}")
print(f"Number of Records: {df.shape[0]:,}")
print(f"Number of Variables: {df.shape[1]}")

# Verify expected record count
expected_records = 503799
if df.shape[0] == expected_records:
    print(f"✓ Record count matches documentation ({expected_records:,})")
else:
    print(f"⚠ Record count: Expected {expected_records:,}, got {df.shape[0]:,}")

print("\n" + "-"*80)
print("COLUMN NAMES:")
print("-"*80)
print(df.columns.tolist())

print("\n" + "-"*80)
print("DATA TYPES:")
print("-"*80)
print(df.dtypes)

print("\n" + "-"*80)
print("FIRST 5 ROWS:")
print("-"*80)
print(df.head())

print("\n" + "-"*80)
print("MISSING DATA SUMMARY (Top 20 columns by missingness):")
print("-"*80)
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percent': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Missing_Percent', ascending=False).head(20)
print(missing_summary.to_string(index=False))

print("\n" + "-"*80)
print("KEY VARIABLE INSPECTION:")
print("-"*80)

# Demographics
print("\n1. AGE:")
if 'AGE' in df.columns:
    print(df['AGE'].describe())
    print(f"Missing: {df['AGE'].isnull().sum()} ({df['AGE'].isnull().sum()/len(df)*100:.1f}%)")
    print(f"Sample values: {df['AGE'].dropna().head(10).tolist()}")
else:
    print("⚠ AGE column not found")

print("\n2. SEX:")
if 'SEX' in df.columns:
    print(df['SEX'].value_counts(dropna=False))
else:
    print("⚠ SEX column not found")

print("\n3. RACE:")
if 'RACE' in df.columns:
    print(df['RACE'].value_counts(dropna=False))
else:
    print("⚠ RACE column not found")

print("\n4. ETHNICITY:")
if 'ETHNICITY' in df.columns:
    print(df['ETHNICITY'].value_counts(dropna=False))
else:
    print("⚠ ETHNICITY column not found")

print("\n5. RACERETH (Combined race/ethnicity):")
if 'RACERETH' in df.columns:
    print(df['RACERETH'].value_counts(dropna=False))
else:
    print("⚠ RACERETH column not found")

# Diagnosis columns
print("\n" + "-"*80)
print("DIAGNOSIS VARIABLES:")
print("-"*80)
dx_cols = [col for col in df.columns if col.startswith('DX') and not col.startswith('DX_TYPE')]
print(f"Found {len(dx_cols)} diagnosis columns")
print(f"Columns: {dx_cols}")

if 'DX1' in df.columns:
    dx1_missing_rate = df['DX1'].isnull().sum() / len(df) * 100
    print(f"\nDX1 (first-listed diagnosis):")
    print(f"  Missing rate: {dx1_missing_rate:.1f}%")
    print(f"  Available: {(~df['DX1'].isnull()).sum():,} records")
    
    # Sample values
    sample_dx = df['DX1'].dropna().head(10)
    print(f"\n  Sample DX1 values:")
    for i, dx in enumerate(sample_dx, 1):
        print(f"    {i}. '{dx}' (type: {type(dx).__name__}, length: {len(str(dx))})")

if 'DX2' in df.columns:
    dx2_missing_rate = df['DX2'].isnull().sum() / len(df) * 100
    print(f"\nDX2 (second diagnosis):")
    print(f"  Missing rate: {dx2_missing_rate:.1f}%")

# Visit context
print("\n" + "-"*80)
print("VISIT CONTEXT VARIABLES:")
print("-"*80)

if 'MONTH' in df.columns:
    print("\nVISIT MONTH distribution:")
    month_dist = df['MONTH'].value_counts().sort_index()
    for month, count in month_dist.items():
        print(f"  Month {month}: {count:,}")

if 'DAY' in df.columns:
    print("\nDAY OF WEEK distribution:")
    day_mapping = {1: 'Sunday', 2: 'Monday', 3: 'Tuesday', 4: 'Wednesday', 
                   5: 'Thursday', 6: 'Friday', 7: 'Saturday'}
    day_dist = df['DAY'].value_counts().sort_index()
    for day, count in day_dist.items():
        print(f"  {day_mapping.get(day, day)}: {count:,}")

# Survey design variables
print("\n" + "-"*80)
print("SURVEY DESIGN VARIABLES:")
print("-"*80)

if 'VISWT' in df.columns:
    print("\nVISWT (visit weight):")
    print(df['VISWT'].describe())
    total_weighted = df['VISWT'].sum()
    print(f"\nSum of weights: {total_weighted:,.0f}")
    print(f"Expected from documentation: 123,817,677")
    diff = abs(total_weighted - 123817677)
    if diff < 1000:
        print(f"✓ Weight sum matches documentation (diff: {diff:,.0f})")
    else:
        print(f"⚠ Weight sum differs by {diff:,.0f}")

if 'HCID_S' in df.columns:
    n_hc = df['HCID_S'].nunique()
    print(f"\nNumber of unique health centers (HCID_S): {n_hc}")
    print(f"Expected: 107")
    if n_hc == 107:
        print("✓ Health center count matches")
    else:
        print(f"⚠ Expected 107, got {n_hc}")

if 'STRATUM_S' in df.columns:
    n_strata = df['STRATUM_S'].nunique()
    print(f"\nNumber of strata (STRATUM_S): {n_strata}")
    print(f"Expected: 8")
    if n_strata == 8:
        print("✓ Strata count matches")

# Check marital status
if 'MARITAL' in df.columns:
    print("\n" + "-"*80)
    print("MARITAL STATUS:")
    print("-"*80)
    print(df['MARITAL'].value_counts(dropna=False))

print("\n" + "="*80)
print("DATA LOADING COMPLETE")
print("="*80)
print(f"✓ Loaded {df.shape[0]:,} records")
print(f"✓ {df.shape[1]} variables")
print(f"✓ Ready for feature engineering")

# Quick data quality summary
print("\n" + "="*80)
print("DATA QUALITY SUMMARY")
print("="*80)
print(f"Total records: {len(df):,}")
if 'DX1' in df.columns:
    print(f"Records with DX1: {(~df['DX1'].isnull()).sum():,} ({(~df['DX1'].isnull()).sum()/len(df)*100:.1f}%)")
if 'AGE' in df.columns and 'SEX' in df.columns:
    print(f"Complete demographic data (AGE & SEX): {df[['AGE', 'SEX']].notna().all(axis=1).sum():,}")

# Save for next steps
output_dir = '/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/data/processed'
os.makedirs(output_dir, exist_ok=True)

df.to_pickle(f'{output_dir}/hc2024_raw.pkl')
print(f"\n✓ Saved: {output_dir}/hc2024_raw.pkl")

# Also save inspection report
report_dir = '/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/outputs'
os.makedirs(report_dir, exist_ok=True)

with open(f'{report_dir}/00_data_inspection_report.txt', 'w') as f:
    f.write("NAMCS HC 2024 Data Inspection Report\n")
    f.write("="*80 + "\n\n")
    f.write(f"Dataset Shape: {df.shape}\n")
    f.write(f"Records: {df.shape[0]:,}\n")
    f.write(f"Variables: {df.shape[1]}\n\n")
    f.write(f"Columns: {df.columns.tolist()}\n\n")
    f.write("Top 20 Missing Data:\n")
    f.write(missing_summary.to_string() + "\n")

print(f"✓ Saved: {report_dir}/00_data_inspection_report.txt")

Loading: /Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/data/raw/HC2024_SAS.sas7bdat
File exists: True
✓ Successfully loaded SAS file

NAMCS HC 2024 DATA - INITIAL INSPECTION

Dataset Shape: (503799, 75)
Number of Records: 503,799
Number of Variables: 75
✓ Record count matches documentation (503,799)

--------------------------------------------------------------------------------
COLUMN NAMES:
--------------------------------------------------------------------------------
['DAY', 'MONTH', 'HCID_S', 'PUF_ID', 'MARITAL', 'AGE', 'AGE_GROUP', 'AGE_U1_MONTH', 'ETHNICITY', 'RACE', 'RACERETH', 'SEX', 'DX1', 'DX2', 'DX3', 'DX4', 'DX5', 'DX6', 'DX7', 'DX8', 'DX9', 'DX10', 'DX11', 'DX12', 'DX13', 'DX14', 'DX15', 'DX16', 'DX17', 'DX18', 'DX19', 'DX20', 'DX21', 'DX22', 'DX23', 'DX24', 'DX25', 'DX26', 'DX27', 'DX28', 'DX29', 'DX30', 'DX_TYPE1', 'DX_TYPE2', 'DX_TYPE3', 'DX_TYPE4', 'DX_TYPE5', 'DX_TYPE6', 'DX_TYPE7', 'DX_TYPE8', 'DX_TYPE9', 'DX_TYPE10', 'DX_TYPE11', '